<a href="https://colab.research.google.com/github/angelrecalde2024/Power-System-Planning-and-Transmission-Design-2026/blob/main/INGP1118_TransmissionExpansion_SecurityConstr_LinearDC_OPF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

LINEAR SECURITY-CONSTRAINED DECOUPLED DC OPTIMAL POWER FLOW

The Security Constrained optimal power flow does the following:


*   OBJECTIVE: minimize the total cost of generation
*   CONSTRAINTS: 1) Decoupled DC power balance on each bus, 2) Generator limits, 3) slack angle, 4) line limits, 5) security constrained power flow
*   Power limits ARE enforced in the transmission lines.



This code performs an optimal power flow considering:


*   Piecewise linear cost function approximation of the quadratic cost function for the generators


**Keep in ming that: There is Preventive SCOPF and Corrective SCOPF**



FIRST STEP

Structural screening (N-1 credibility): if a single-line outage islands the system → discard that contingency (not “credible” for preventive SCOPF unless you explicitly model island controls, multiple slack buses, UFLS, etc.).



*   islands? → discard
*   connected? → run a DC feasibility LP with generator limits + line limits (ENS=0). If feasible → keep, else discard
*   prints a list of infeasible contingencies and the feasible set you’ll pass to SCOPF.

In [1]:
# =====================================================
# CELL 1: INSTALL + DATA + N-1 SCREENING
# =====================================================
!pip -q install pyomo highspy pypower openpyxl pandas numpy

import numpy as np
import pandas as pd
import pyomo.environ as pyo

from pypower.idx_bus import BUS_I, PD
from pypower.idx_gen import GEN_BUS, PMIN, PMAX
from pypower.idx_brch import F_BUS, T_BUS, BR_X, BR_STATUS
from pypower.makeBdc import makeBdc

# -----------------------
# Settings
# -----------------------
EXCEL = "/content/Data6busSystemTEP.xlsx"   # Colab: "/content/Data6busSystemTEP.xlsx"
BASE_MVA = 100.0
SLACK_BUS_EXT = 1

# -----------------------
# Read Excel (exact headers)
# -----------------------
gen_df  = pd.read_excel(EXCEL, sheet_name="generator_relaxed")
load_df = pd.read_excel(EXCEL, sheet_name="load")
line_df = pd.read_excel(EXCEL, sheet_name="lines_relaxed")

req_gen  = {"generator bus","a","b","c","Pmin","Pmax"}
req_load = {"load bus","Pload"}
req_line = {"from","to","X","Flow limit"}

if not req_gen.issubset(set(gen_df.columns)):
    raise ValueError(f"'generator' headers mismatch. Found: {list(gen_df.columns)}")
if not req_load.issubset(set(load_df.columns)):
    raise ValueError(f"'load' headers mismatch. Found: {list(load_df.columns)}")
if not req_line.issubset(set(line_df.columns)):
    raise ValueError(f"'lines' headers mismatch. Found: {list(line_df.columns)}")

# -----------------------
# Build internal (0-based) case
# -----------------------
buses_ext = sorted(
    set(gen_df["generator bus"].astype(int))
    | set(load_df["load bus"].astype(int))
    | set(line_df["from"].astype(int))
    | set(line_df["to"].astype(int))
)
nb = len(buses_ext)
if SLACK_BUS_EXT not in buses_ext:
    raise ValueError(f"Slack bus {SLACK_BUS_EXT} not found in buses: {buses_ext}")

ext2int = {b:i for i,b in enumerate(buses_ext)}   # 0..nb-1
int2ext = {i:b for b,i in ext2int.items()}
slack_i = ext2int[SLACK_BUS_EXT]

# bus matrix
bus = np.zeros((nb, 13), dtype=float)
for i in range(nb):
    bus[i, BUS_I] = i
for _, r in load_df.iterrows():
    bus[ext2int[int(r["load bus"])], PD] += float(r["Pload"])
Pd = bus[:, PD].copy()

# gen matrix (minimal fields + P limits)
ng = len(gen_df)
gen = np.zeros((ng, 10), dtype=float)
for g, (_, r) in enumerate(gen_df.iterrows()):
    gen[g, GEN_BUS] = ext2int[int(r["generator bus"])]
    gen[g, PMIN] = float(r["Pmin"])
    gen[g, PMAX] = float(r["Pmax"])
gen_bus = gen[:, GEN_BUS].astype(int)
Pmin = gen[:, PMIN].copy()
Pmax = gen[:, PMAX].copy()

# branch matrix + line params
nl = len(line_df)
branch = np.zeros((nl, 13), dtype=float)
f_bus = np.zeros(nl, dtype=int)
t_bus = np.zeros(nl, dtype=int)
x = np.zeros(nl, dtype=float)
fmax = np.zeros(nl, dtype=float)

for l, (_, r) in enumerate(line_df.iterrows()):
    f = ext2int[int(r["from"])]
    t = ext2int[int(r["to"])]
    f_bus[l] = f
    t_bus[l] = t
    x[l] = float(r["X"])
    fmax[l] = float(r["Flow limit"])

    branch[l, F_BUS] = f
    branch[l, T_BUS] = t
    branch[l, BR_X] = x[l]
    branch[l, BR_STATUS] = 1.0

print("Data check:")
print(f"  buses={nb}, gens={ng}, lines={nl}")
print(f"  total load={Pd.sum():.3f} MW, sum Pmax={Pmax.sum():.3f} MW")
print(f"  slack bus ext={SLACK_BUS_EXT} -> int={slack_i}")

# -----------------------
# Graph utilities for islanding check
# -----------------------
def connected_after_outage(out_line_idx: int | None) -> bool:
    # Build adjacency excluding the outaged line
    adj = {i:set() for i in range(nb)}
    for l in range(nl):
        if out_line_idx is not None and l == out_line_idx:
            continue
        u, v = int(f_bus[l]), int(t_bus[l])
        adj[u].add(v)
        adj[v].add(u)

    # BFS from slack
    visited = set()
    stack = [slack_i]
    while stack:
        u = stack.pop()
        if u in visited:
            continue
        visited.add(u)
        for v in adj[u]:
            if v not in visited:
                stack.append(v)

    # connected iff all buses reachable from slack
    return len(visited) == nb

# -----------------------
# DC feasibility check under limits (ENS=0)
# For each scenario, solve a *feasibility LP*:
#  - Pg within [Pmin,Pmax]
#  - nodal balance (DC)
#  - line limits
# -----------------------
def dc_feasible_under_limits(out_line_idx: int | None):
    # Quick structural check first
    if not connected_after_outage(out_line_idx):
        return (False, "islanding", None)

    # Build Bbus
    br = branch.copy()
    if out_line_idx is not None:
        br[out_line_idx, BR_STATUS] = 0.0

    try:
        Bbus, _, _, _ = makeBdc(BASE_MVA, bus, br)
        Bbus = np.array(Bbus.todense())
    except Exception as e:
        return (False, f"makeBdc error: {e}", None)

    # Build feasibility LP
    m = pyo.ConcreteModel()
    m.B = pyo.RangeSet(0, nb-1)
    m.G = pyo.RangeSet(0, ng-1)
    m.L = pyo.RangeSet(0, nl-1)

    m.theta = pyo.Var(m.B)
    m.Pg = pyo.Var(m.G)

    m.theta[slack_i].fix(0.0)

    def gb(mdl, g):
        return (float(Pmin[g]), mdl.Pg[g], float(Pmax[g]))
    m.gb = pyo.Constraint(m.G, rule=gb)

    def bal(mdl, i):
        Pg_i = sum(mdl.Pg[g] for g in mdl.G if int(gen_bus[g]) == i)
        net = BASE_MVA * sum(float(Bbus[i, j]) * mdl.theta[j] for j in range(nb))
        return Pg_i - float(Pd[i]) == net
    m.balance = pyo.Constraint(m.B, rule=bal)

    def flow_expr(mdl, l):
        return BASE_MVA * (mdl.theta[int(f_bus[l])] - mdl.theta[int(t_bus[l])]) / float(x[l])
    m.F = pyo.Expression(m.L, rule=flow_expr)

    def ub(mdl, l):
        if out_line_idx is not None and l == out_line_idx:
            return pyo.Constraint.Skip
        return m.F[l] <= float(fmax[l])

    def lb(mdl, l):
        if out_line_idx is not None and l == out_line_idx:
            return pyo.Constraint.Skip
        return -m.F[l] <= float(fmax[l])

    m.ub = pyo.Constraint(m.L, rule=ub)
    m.lb = pyo.Constraint(m.L, rule=lb)

    if out_line_idx is not None:
        m.out0 = pyo.Constraint(expr=m.F[out_line_idx] == 0.0)

    m.obj = pyo.Objective(expr=0.0)

    solver = pyo.SolverFactory("highs")
    solver.options["presolve"] = "on"

    try:
        res = solver.solve(m, tee=False, load_solutions=False)

        status = str(res.solver.status).lower()
        term = str(res.solver.termination_condition).lower()

        if "optimal" in term or "feasible" in term:
            m.solutions.load_from(res)
            Pg_sol = np.array([pyo.value(m.Pg[g]) for g in range(ng)])
            return (True, "feasible", Pg_sol)

        elif "infeasible" in term:
            return (False, "dc infeasible under limits", None)

        else:
            return (False, f"solver status={res.solver.status}", None)

    except Exception as e:
        # Catch HiGHS internal NoFeasibleSolutionError
        return (False, f"solver exception: {e}", None)

# -----------------------
# Screen base + all N-1
# -----------------------
feasible_contingencies = []   # list of outaged line indices that are feasible
infeasible_contingencies = [] # list of (line idx, reason)

dispatch_results = []

# Base case
ok0, reason0, Pg0 = dc_feasible_under_limits(out_line_idx=None)
print("\nBase case feasibility:", ok0, "-", reason0)

if ok0:
    dispatch_results.append(("BASE", None, Pg0))
else:
    raise RuntimeError("Base case infeasible")

# N-1 checks
for out_l in range(nl):
    ok, reason, Pg_sol = dc_feasible_under_limits(out_line_idx=out_l)
    if ok:
        feasible_contingencies.append(out_l)
        dispatch_results.append((f"OUT_{out_l}", out_l, Pg_sol))
    else:
        infeasible_contingencies.append((out_l, reason))

print("\n==== N-1 Screening Summary ====")
print(f"Feasible contingencies: {len(feasible_contingencies)} / {nl}")
print(f"Infeasible contingencies: {len(infeasible_contingencies)} / {nl}")

print("\n===== DISPATCH PER FEASIBLE SCENARIO =====")

rows = []
for label, out_l, Pg_sol in dispatch_results:
    row = {"Scenario": label}
    if out_l is None:
        row["OutagedLine"] = "BASE"
    else:
        row["OutagedLine"] = f"{line_df.iloc[out_l]['from']} - {line_df.iloc[out_l]['to']}"

    for g in range(ng):
        row[f"Pg_gen{g}"] = Pg_sol[g]

    rows.append(row)

dispatch_table = pd.DataFrame(rows)
display(dispatch_table)

if infeasible_contingencies:
    df_bad = pd.DataFrame(infeasible_contingencies, columns=["OutagedLineIndex", "Reason"])
    df_bad["Out_from"] = df_bad["OutagedLineIndex"].apply(lambda l: int(line_df.iloc[l]["from"]))
    df_bad["Out_to"]   = df_bad["OutagedLineIndex"].apply(lambda l: int(line_df.iloc[l]["to"]))
    display(df_bad)

# We will pass these to Cell 2:
# - feasible_contingencies
# - the internal case data (bus/gen/branch/etc.)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.6/347.6 kB 5.4 MB/s eta 0:00:00
Data check:
  buses=6, gens=3, lines=11
  total load=300.000 MW, sum Pmax=1060.000 MW
  slack bus ext=1 -> int=0

Base case feasibility: True - feasible

==== N-1 Screening Summary ====
Feasible contingencies: 4 / 11
Infeasible contingencies: 7 / 11

===== DISPATCH PER FEASIBLE SCENARIO =====


,Scenario,OutagedLine,Pg_gen0,Pg_gen1,Pg_gen2
0,BASE,BASE,50.000000,37.500000,212.500000
1,OUT_0,1.0 - 2.0,50.000000,53.834423,196.165577
2,OUT_3,2.0 - 3.0,174.441170,37.500000,88.058830
3,OUT_9,4.0 - 5.0,102.632861,37.500000,159.867139
4,OUT_10,5.0 - 6.0,84.211601,170.788399,45.000000


,OutagedLineIndex,Reason,Out_from,Out_to
0,1,solver exception: Cannot load a SolverResults ...,1,4
1,2,solver exception: Cannot load a SolverResults ...,1,5
2,4,solver exception: Cannot load a SolverResults ...,2,4
3,5,solver exception: Cannot load a SolverResults ...,2,5
4,6,solver exception: Cannot load a SolverResults ...,2,6
5,7,solver exception: Cannot load a SolverResults ...,3,5
6,8,solver exception: Cannot load a SolverResults ...,3,6


STEP 2

Operational screening (DC feasibility under limits): even if connected, the outage may make the network unable to serve all loads within generator Pmax and line limits with ENS = 0. Those scenarios are “credible but infeasible” under our rules, so you discard them from the SC set and proceed with the remaining set.
This prevents the SCOPF from being dominated by a few structurally impossible cases, and it avoids the false impression that “the optimization is broken.”



*   PWL costs
*   line limits per scenario
*   base-case LMPs
*   scenario-wise line limit flags
*   worst-contingency ENS summary (for completeness; if you set ENS_MAX=0, ENS will be 0)


**SCOPF: PREVENTIVE**

In [2]:
# =====================================================
# CELL 2: Preventive SC-DCOPF over feasible contingencies
# =====================================================

# -----------------------
# SCOPF settings
# -----------------------
K_SEG = 5
ENS_MAX = 500.0   # hard N-1 over the *feasible* set; increase to relax globally (MW)

# -----------------------
# Build scenario list:
# s=0 base
# s=1..S where each s corresponds to feasible_contingencies[s-1]
# -----------------------
scenario_outage = {0: None}
for s, out_l in enumerate(feasible_contingencies, start=1):
    scenario_outage[s] = out_l
S_list = sorted(scenario_outage.keys())  # [0,1,2,...]

# -----------------------
# Precompute Bbus for each scenario
# -----------------------
def Bbus_for_outage(out_l: int | None):
    br = branch.copy()
    if out_l is not None:
        br[out_l, BR_STATUS] = 0.0
    Bbus, _, _, _ = makeBdc(BASE_MVA, bus, br)
    return np.array(Bbus.todense())

Bbus_s = {s: Bbus_for_outage(scenario_outage[s]) for s in S_list}

# -----------------------
# PWL parameters (equal segments)
# -----------------------
a = gen_df["a"].to_numpy(dtype=float)
b = gen_df["b"].to_numpy(dtype=float)
c = gen_df["c"].to_numpy(dtype=float)

def build_pwl(Pmin, Pmax, a, b, c, K):
    dP = np.zeros((ng, K))
    slope = np.zeros((ng, K))
    for g in range(ng):
        pts = np.linspace(Pmin[g], Pmax[g], K+1)
        dP[g,:] = np.diff(pts)
        C = a[g] + b[g]*pts + c[g]*(pts**2)
        slope[g,:] = np.diff(C) / dP[g,:]
    return dP, slope

dP_seg, slope_seg = build_pwl(Pmin, Pmax, a, b, c, K_SEG)

# gens-at-bus list
gens_at_bus = {i: [] for i in range(nb)}
for g in range(ng):
    gens_at_bus[int(gen_bus[g])].append(g)

# -----------------------
# Build SCOPF model (preventive Pg shared across scenarios)
# -----------------------
m = pyo.ConcreteModel()
m.S = pyo.Set(initialize=S_list, ordered=True)
m.B = pyo.RangeSet(0, nb-1)
m.G = pyo.RangeSet(0, ng-1)
m.K = pyo.RangeSet(0, K_SEG-1)
m.L = pyo.RangeSet(0, nl-1)

m.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)

# Vars
m.theta = pyo.Var(m.B, m.S)
m.xseg  = pyo.Var(m.G, m.K, domain=pyo.NonNegativeReals)
m.ENS   = pyo.Var(m.B, m.S, domain=pyo.NonNegativeReals)

# Slack angle per scenario
for s in S_list:
    m.theta[slack_i, s].fix(0.0)

# ENS in base fixed to 0 (optional; consistent with your previous approach)
m.ENS_base0 = pyo.ConstraintList()
for i in range(nb):
    m.ENS_base0.add(m.ENS[i, 0] == 0.0)

# Segment bounds
def seg_ub(mdl, g, k):
    return mdl.xseg[g, k] <= float(dP_seg[g, k])
m.seg_ub = pyo.Constraint(m.G, m.K, rule=seg_ub)

# Preventive Pg
def Pg_rule(mdl, g):
    return float(Pmin[g]) + sum(mdl.xseg[g, k] for k in mdl.K)
m.Pg = pyo.Expression(m.G, rule=Pg_rule)

# Balance per bus per scenario
def balance(mdl, i, s):
    Pg_i = sum(mdl.Pg[g] for g in gens_at_bus[i]) if gens_at_bus[i] else 0.0
    rhs = BASE_MVA * sum(float(Bbus_s[s][i, j]) * mdl.theta[j, s] for j in range(nb))
    return Pg_i - float(Pd[i]) + mdl.ENS[i, s] == rhs
m.balance = pyo.Constraint(m.B, m.S, rule=balance)

# Line flows
def flow_expr(mdl, l, s):
    return BASE_MVA * (mdl.theta[int(f_bus[l]), s] - mdl.theta[int(t_bus[l]), s]) / float(x[l])
m.F = pyo.Expression(m.L, m.S, rule=flow_expr)

# Line limits, skipping outaged line in each scenario; also enforce outaged flow=0
m.flow_ub = pyo.ConstraintList()
m.flow_lb = pyo.ConstraintList()
m.out0 = pyo.ConstraintList()

for s in S_list:
    out_l = scenario_outage[s]
    for l in range(nl):
        if out_l is not None and l == out_l:
            # outaged: set flow=0
            m.out0.add(m.F[l, s] == 0.0)
        else:
            m.flow_ub.add(m.F[l, s] <= float(fmax[l]))
            m.flow_lb.add(-m.F[l, s] <= float(fmax[l]))

# Global ENS budget across contingencies only (exclude base s=0)
def ens_budget(mdl):
    return sum(mdl.ENS[i, s] for i in range(nb) for s in S_list if s != 0) <= float(ENS_MAX)
m.ens_budget = pyo.Constraint(rule=ens_budget)

# Objective: PWL generation cost
m.obj = pyo.Objective(
    expr=sum(float(slope_seg[g, k]) * m.xseg[g, k] for g in range(ng) for k in range(K_SEG)),
    sense=pyo.minimize
)

# Solve safely
solver = pyo.SolverFactory("highs")
res = solver.solve(m, tee=False, load_solutions=False)
term = str(res.solver.termination_condition).lower()
print("Solver:", res.solver.status, res.solver.termination_condition)
if "optimal" not in term and "feasible" not in term:
    raise RuntimeError("SCOPF not solved. Try increasing ENS_MAX or inspect feasible_contingencies.")
m.solutions.load_from(res)

# -----------------------
# Extract outputs
# -----------------------
Pg = np.array([pyo.value(m.Pg[g]) for g in range(ng)], dtype=float)

theta = {s: np.array([pyo.value(m.theta[i, s]) for i in range(nb)], dtype=float) for s in S_list}
ENS   = {s: np.array([pyo.value(m.ENS[i, s]) for i in range(nb)], dtype=float) for s in S_list}
Fflow = {(l, s): float(pyo.value(m.F[l, s])) for s in S_list for l in range(nl)}

# Base-case LMP from duals of balance constraints at s=0
LMP_base = np.array([-m.dual[m.balance[i, 0]] for i in range(nb)], dtype=float)

# -----------------------
# Reporting: dispatch + cost reporting (quadratic) + total cost
# -----------------------
gen_rows = []
total_cost = 0.0
for g in range(ng):
    P = Pg[g]
    Cg = float(a[g] + b[g]*P + c[g]*(P**2))
    total_cost += Cg
    gen_rows.append([g, int2ext[int(gen_bus[g])], P, Cg])

gen_table = pd.DataFrame(gen_rows, columns=["Gen", "Bus", "Pg (MW)", "Quadratic cost a+bP+cP^2 ($/h)"])
print("\n===== GENERATOR DISPATCH =====")
display(gen_table)
print(f"TOTAL SYSTEM COST (reported quadratic) = {total_cost:.2f} $/h")

# Base-case LMP table
lmp_table = pd.DataFrame({
    "Bus": [int2ext[i] for i in range(nb)],
    "Theta_base (rad)": theta[0],
    "LMP_base ($/MW)": LMP_base
})
print("\n===== BASE-CASE BUS ANGLES + LMP =====")
display(lmp_table)

# Scenario-wise line limit flags (for kept scenarios only)
tol = 1e-6
rows = []
for s in S_list:
    out_l = scenario_outage[s]
    scen_label = "BASE" if s == 0 else f"OUTAGE line {out_l} (kept)"
    for l in range(nl):
        flow = Fflow[(l, s)]
        lim = float(fmax[l])
        if out_l is not None and l == out_l:
            within = "OUTAGED"
        else:
            within = "YES" if abs(flow) <= lim + tol else "NO"
        rows.append([s, scen_label, l, int2ext[int(f_bus[l])], int2ext[int(t_bus[l])], flow, lim, within])

line_flags = pd.DataFrame(rows, columns=["Scenario","ScenarioLabel","Line","From","To","Flow (MW)","Limit (MW)","Within?"])
print("\n===== SCENARIO-WISE LINE LIMIT FLAGS (kept scenarios) =====")
display(line_flags)

# Worst-contingency ENS summary among kept contingencies
ens_sum_rows = []
for s in S_list:
    if s == 0:
        continue
    out_l = scenario_outage[s]
    ens_total = float(np.sum(ENS[s]))
    ens_sum_rows.append([s, out_l, int(line_df.iloc[out_l]["from"]), int(line_df.iloc[out_l]["to"]), ens_total])

ens_table = pd.DataFrame(ens_sum_rows, columns=["Scenario","OutagedLineIndex","Out_from","Out_to","Total ENS (MW)"])
if len(ens_table) > 0:
    worst = ens_table.sort_values("Total ENS (MW)", ascending=False).head(1)
    print("\n===== WORST-CONTINGENCY ENS SUMMARY (kept) =====")
    display(worst)
    print("\n(All kept contingencies ENS totals)")
    display(ens_table.sort_values("Total ENS (MW)", ascending=False))
else:
    print("\nNo contingencies were kept as feasible in screening; SCOPF ran only base case.")

Solver: error infeasible


ValueError: Cannot load a SolverResults object with bad status: error

At this point, you may have gotten the following error:


*   Solver: error infeasible

In the screening test, for each contingency we asked: Is there some dispatch Pg that works for this topology?

But preventive SCOPF requires: Is there one single dispatch Pg that works for all topologies simultaneously?

If at this point we get an infeasibility error, the syste, is not N-1 secure under preventive dispatch.

Even though each contingency individually is feasible with its own redispatch, there may not exist a common preventive dispatch that keeps all post-contingency flows within limits.

This is exactly why,



*   Preventive SCOPF is conservative
*   Corrective SCOPF is widely used in practice.

**Now we do Corrective SCOPF**






In [3]:
# =====================================================
# CELL 3: Corrective SCOPF (redispatch per contingency)
# Uses feasible_contingencies from Cell 1
# =====================================================

import numpy as np
import pandas as pd
import pyomo.environ as pyo
from pypower.makeBdc import makeBdc
from pypower.idx_brch import BR_STATUS

# -----------------------
# User knobs
# -----------------------
K_SEG = 5

# Corrective redispatch capability:
# RAMP_FRAC = 0  -> no redispatch (reduces to preventive-like)
# RAMP_FRAC = 1  -> full range allowed (Pg can move anywhere within [Pmin,Pmax] per scenario)
RAMP_FRAC = 1.0

# -----------------------
# Scenario mapping
# s=0 base
# s=1..S outages from feasible_contingencies
# -----------------------
scenario_outage = {0: None}
for s, out_l in enumerate(feasible_contingencies, start=1):
    scenario_outage[s] = out_l
S_list = sorted(scenario_outage.keys())
S = len(S_list) - 1

print("Corrective SCOPF scenarios (including base):", len(S_list))
print("Outages kept:", feasible_contingencies)

# -----------------------
# Build Bbus per scenario
# -----------------------
def Bbus_for_outage(out_l):
    br = branch.copy()
    if out_l is not None:
        br[out_l, BR_STATUS] = 0.0
    Bbus, _, _, _ = makeBdc(BASE_MVA, bus, br)
    return np.array(Bbus.todense())

Bbus_s = {s: Bbus_for_outage(scenario_outage[s]) for s in S_list}

# -----------------------
# PWL parameters for quadratic cost a+bP+cP^2
# (we use base-case cost; contingencies are feasibility/redispatch)
# -----------------------
a = gen_df["a"].to_numpy(dtype=float)
b = gen_df["b"].to_numpy(dtype=float)
c = gen_df["c"].to_numpy(dtype=float)

def build_pwl(Pmin, Pmax, a, b, c, K):
    dP = np.zeros((ng, K))
    slope = np.zeros((ng, K))
    for g in range(ng):
        pts = np.linspace(Pmin[g], Pmax[g], K+1)
        dP[g,:] = np.diff(pts)
        C = a[g] + b[g]*pts + c[g]*(pts**2)
        slope[g,:] = np.diff(C) / dP[g,:]
    return dP, slope

dP_seg, slope_seg = build_pwl(Pmin, Pmax, a, b, c, K_SEG)

# Precompute gens-at-bus
gens_at_bus = {i: [] for i in range(nb)}
for g in range(ng):
    gens_at_bus[int(gen_bus[g])].append(g)

# Redispatch bounds (MW)
R = RAMP_FRAC * (Pmax - Pmin)

# =====================================================
# MODEL
# =====================================================
m = pyo.ConcreteModel()
m.S = pyo.Set(initialize=S_list, ordered=True)
m.B = pyo.RangeSet(0, nb-1)
m.G = pyo.RangeSet(0, ng-1)
m.K = pyo.RangeSet(0, K_SEG-1)
m.L = pyo.RangeSet(0, nl-1)

m.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)

# Base-case PWL segment vars -> Pg0
m.xseg0 = pyo.Var(m.G, m.K, domain=pyo.NonNegativeReals)

def seg_ub0(mdl, g, k):
    return mdl.xseg0[g, k] <= float(dP_seg[g, k])
m.seg_ub0 = pyo.Constraint(m.G, m.K, rule=seg_ub0)

def Pg0_rule(mdl, g):
    return float(Pmin[g]) + sum(mdl.xseg0[g, k] for k in mdl.K)
m.Pg0 = pyo.Expression(m.G, rule=Pg0_rule)

# Scenario generator outputs Pg[g,s]
m.Pg = pyo.Var(m.G, m.S)

# Angles by scenario
m.theta = pyo.Var(m.B, m.S)

# Slack angle in every scenario
for s in S_list:
    m.theta[slack_i, s].fix(0.0)

# Link base scenario Pg to Pg0
def base_link(mdl, g):
    return mdl.Pg[g, 0] == mdl.Pg0[g]
m.base_link = pyo.Constraint(m.G, rule=base_link)

# Generator limits for all scenarios
def gen_bounds(mdl, g, s):
    return (float(Pmin[g]), mdl.Pg[g, s], float(Pmax[g]))
m.gen_bounds = pyo.Constraint(m.G, m.S, rule=gen_bounds)

# Corrective redispatch bounds for contingencies
# |Pg[g,s] - Pg0[g]| <= R[g]
def ramp_up(mdl, g, s):
    if s == 0:
        return pyo.Constraint.Skip
    return mdl.Pg[g, s] - mdl.Pg0[g] <= float(R[g])

def ramp_dn(mdl, g, s):
    if s == 0:
        return pyo.Constraint.Skip
    return mdl.Pg0[g] - mdl.Pg[g, s] <= float(R[g])

m.ramp_up = pyo.Constraint(m.G, m.S, rule=ramp_up)
m.ramp_dn = pyo.Constraint(m.G, m.S, rule=ramp_dn)

# Power balance per bus per scenario (ENS=0 here)
def balance(mdl, i, s):
    Pg_i = sum(mdl.Pg[g, s] for g in gens_at_bus[i]) if gens_at_bus[i] else 0.0
    rhs = BASE_MVA * sum(float(Bbus_s[s][i, j]) * mdl.theta[j, s] for j in range(nb))
    return Pg_i - float(Pd[i]) == rhs

m.balance = pyo.Constraint(m.B, m.S, rule=balance)

# Line flows + limits (skip outaged line in scenario)
def flow_expr(mdl, l, s):
    return BASE_MVA * (mdl.theta[int(f_bus[l]), s] - mdl.theta[int(t_bus[l]), s]) / float(x[l])
m.F = pyo.Expression(m.L, m.S, rule=flow_expr)

m.flow_ub = pyo.ConstraintList()
m.flow_lb = pyo.ConstraintList()
m.out0 = pyo.ConstraintList()

for s in S_list:
    out_l = scenario_outage[s]
    for l in range(nl):
        if out_l is not None and l == out_l:
            m.out0.add(m.F[l, s] == 0.0)
        else:
            m.flow_ub.add(m.F[l, s] <= float(fmax[l]))
            m.flow_lb.add(-m.F[l, s] <= float(fmax[l]))

# Objective: base-case generation cost (PWL)
m.obj = pyo.Objective(
    expr=sum(float(slope_seg[g, k]) * m.xseg0[g, k] for g in range(ng) for k in range(K_SEG)),
    sense=pyo.minimize
)

# =====================================================
# SOLVE
# =====================================================
solver = pyo.SolverFactory("highs")
res = solver.solve(m, tee=False, load_solutions=False)
term = str(res.solver.termination_condition).lower()
print("Solver:", res.solver.status, res.solver.termination_condition)
if "optimal" not in term and "feasible" not in term:
    raise RuntimeError("Corrective SCOPF not solved.")
m.solutions.load_from(res)

# =====================================================
# EXTRACT RESULTS
# =====================================================
Pg0 = np.array([pyo.value(m.Pg0[g]) for g in range(ng)], dtype=float)
Pg_s = {s: np.array([pyo.value(m.Pg[g, s]) for g in range(ng)], dtype=float) for s in S_list}

theta0 = np.array([pyo.value(m.theta[i, 0]) for i in range(nb)], dtype=float)
LMP_base = np.array([-m.dual[m.balance[i, 0]] for i in range(nb)], dtype=float)

# Generator cost reporting (quadratic) for base dispatch
total_cost = float(np.sum(a + b*Pg0 + c*(Pg0**2)))
gen_rows = []
for g in range(ng):
    gen_rows.append([g, int2ext[int(gen_bus[g])], Pg0[g], float(a[g] + b[g]*Pg0[g] + c[g]*(Pg0[g]**2))])
gen_table = pd.DataFrame(gen_rows, columns=["Gen","Bus","Pg0 (MW)","Quadratic cost ($/h)"])

print("\n===== BASE DISPATCH (Corrective SCOPF) =====")
display(gen_table)
print(f"TOTAL SYSTEM COST (reported quadratic) = {total_cost:.2f} $/h")

print("\n===== BASE-CASE LMP TABLE =====")
lmp_table = pd.DataFrame({
    "Bus": [int2ext[i] for i in range(nb)],
    "Theta_base (rad)": theta0,
    "LMP_base ($/MW)": LMP_base
})
display(lmp_table)

# Redispatch table across scenarios
rows = []
for s in S_list:
    if s == 0:
        label = "BASE"
    else:
        out_l = scenario_outage[s]
        label = f"OUT_{out_l} ({int(line_df.iloc[out_l]['from'])}-{int(line_df.iloc[out_l]['to'])})"
    row = {"Scenario": s, "Label": label}
    for g in range(ng):
        row[f"Pg_gen{g}"] = Pg_s[s][g]
        row[f"dPg_gen{g}"] = Pg_s[s][g] - Pg0[g]
    rows.append(row)

dispatch_scen_table = pd.DataFrame(rows)
print("\n===== SCENARIO DISPATCH + REDISPATCH (Corrective) =====")
display(dispatch_scen_table)

# Scenario-wise line limit flags
tol = 1e-6
flag_rows = []
for s in S_list:
    out_l = scenario_outage[s]
    scen_label = "BASE" if s == 0 else f"OUTAGE line {out_l}"
    for l in range(nl):
        flow = float(pyo.value(m.F[l, s]))
        lim = float(fmax[l])
        if out_l is not None and l == out_l:
            within = "OUTAGED"
        else:
            within = "YES" if abs(flow) <= lim + tol else "NO"
        flag_rows.append([s, scen_label, l, int2ext[int(f_bus[l])], int2ext[int(t_bus[l])], flow, lim, within])

line_flags = pd.DataFrame(flag_rows, columns=["Scenario","ScenarioLabel","Line","From","To","Flow (MW)","Limit (MW)","Within?"])
print("\n===== SCENARIO-WISE LINE LIMIT FLAGS (Corrective) =====")
display(line_flags)

Corrective SCOPF scenarios (including base): 5
Outages kept: [0, 3, 9, 10]
Solver: ok optimal

===== BASE DISPATCH (Corrective SCOPF) =====


,Gen,Bus,Pg0 (MW),Quadratic cost ($/h)
0,0,1,50.0,809.87500
1,1,2,142.0,1846.54396
2,2,3,108.0,1496.39424


TOTAL SYSTEM COST (reported quadratic) = 4152.81 $/h

===== BASE-CASE LMP TABLE =====


,Bus,Theta_base (rad),LMP_base ($/MW)
0,1,0.000000,-12.399925
1,2,0.012001,-12.399925
2,3,0.014804,-12.399925
3,4,-0.061176,12.399925
4,5,-0.076238,12.399925
5,6,-0.057059,12.399925



===== SCENARIO DISPATCH + REDISPATCH (Corrective) =====


,Scenario,Label,Pg_gen0,dPg_gen0,Pg_gen1,dPg_gen1,Pg_gen2,dPg_gen2
0,0,BASE,50.000000,0.000000,142.000000,0.000000,108.000000,0.000000
1,1,OUT_0 (1-2),50.000000,0.000000,53.834423,-88.165577,196.165577,88.165577
2,2,OUT_3 (2-3),174.441170,124.441170,37.500000,-104.500000,88.058830,-19.941170
3,3,OUT_9 (4-5),102.632861,52.632861,37.500000,-104.500000,159.867139,51.867139
4,4,OUT_10 (5-6),84.211601,34.211601,170.788399,28.788399,45.000000,-63.000000



===== SCENARIO-WISE LINE LIMIT FLAGS (Corrective) =====


,Scenario,ScenarioLabel,Line,From,To,Flow (MW),Limit (MW),Within?
0,0,BASE,0,1,2,-6.000633e+00,200.0,YES
1,0,BASE,1,1,4,3.058805e+01,200.0,YES
2,0,BASE,2,1,5,2.541258e+01,200.0,YES
3,0,BASE,3,2,3,-1.121083e+00,120.0,YES
4,0,BASE,4,2,4,7.317736e+01,120.0,YES
5,0,BASE,5,2,5,2.941301e+01,120.0,YES
6,0,BASE,6,2,6,3.453008e+01,120.0,YES
7,0,BASE,7,3,5,3.501605e+01,120.0,YES
8,0,BASE,8,3,6,7.186287e+01,120.0,YES
9,0,BASE,9,4,5,3.765413e+00,120.0,YES


CORRECTIVE SCOPF

In this case, the generation dispatch is different every time there is a line outage. This is reasonable since the dispatch could not remain the same due to network topology reconfiguration.

Up to now, we have a set of contingency dispatch for all feasible scenarios of N-1. Now we could add expected redispatch cost to the system.

We have defined a linearized operating cost for each scenario using the same PWL approximation of the quadratic generation cost curve.

$$
C_s^{\text {pwl }}=\sum_g \sum_k m_{g k} x_{g k}^{(s)} \quad \text { (constants omitted) }
$$


We can also define a contingency probability which is uniform by default.

The expected redispatch cost is:

$$
\mathbb{E}[\Delta C]=\sum_{s>0} p_s\left(C_s^{p w l}-C_0^{p w l}\right)
$$


Then, the total objective becomes:


$$
\min C_0^{\text {pwl }}+\alpha \sum_{s>0} p_s\left(C_s^{\text {pwl }}-C_0^{\text {pwl }}\right)
$$

$\alpha$ equals to 1 means you genuinely value expected corrective operation cost. If $\alpha$ is 0, you recover your previous 'minimize base cost only', that is, the preventive SCOPF.

This stays linear if $C_0^{\text {pwl }}$ is built with segment variables per scenario.


In [6]:
# =====================================================
# CELL 4: Corrective SCOPF (redispatch per contingency)
# Uses feasible_contingencies from Cell 1
# Plus Expected Redispath Cost (when outages occur)
# =====================================================

import numpy as np
import pandas as pd
import pyomo.environ as pyo
from pypower.makeBdc import makeBdc
from pypower.idx_brch import BR_STATUS

# -----------------------
# User knobs
# -----------------------
K_SEG = 5

# Corrective redispatch capability:
# RAMP_FRAC = 0  -> no redispatch (reduces to preventive-like)
# RAMP_FRAC = 1  -> full range allowed (Pg can move anywhere within [Pmin,Pmax] per scenario)
K_SEG = 5
RAMP_FRAC = 1.0

# Expected redispatch weighting
ALPHA_RED = 1.0   # 0.0 -> ignore redispatch cost, 1.0 -> include expected redispatch cost

# Contingency probabilities (uniform by default)
# If you want custom, set PROB = {scenario_id: prob, ...} for s>0
PROB = None

# -----------------------
# Scenario mapping
# s=0 base
# s=1..S outages from feasible_contingencies
# -----------------------
scenario_outage = {0: None}
for s, out_l in enumerate(feasible_contingencies, start=1):
    scenario_outage[s] = out_l
S_list = sorted(scenario_outage.keys())
S = len(S_list) - 1

print("Corrective SCOPF scenarios (including base):", len(S_list))
print("Outages kept:", feasible_contingencies)

# -----------------------
# Build Bbus per scenario
# -----------------------
def Bbus_for_outage(out_l):
    br = branch.copy()
    if out_l is not None:
        br[out_l, BR_STATUS] = 0.0
    Bbus, _, _, _ = makeBdc(BASE_MVA, bus, br)
    return np.array(Bbus.todense())

Bbus_s = {s: Bbus_for_outage(scenario_outage[s]) for s in S_list}

# -----------------------
# PWL parameters for quadratic cost a+bP+cP^2
# (we use base-case cost; contingencies are feasibility/redispatch)
# -----------------------
a = gen_df["a"].to_numpy(dtype=float)
b = gen_df["b"].to_numpy(dtype=float)
c = gen_df["c"].to_numpy(dtype=float)

def build_pwl(Pmin, Pmax, a, b, c, K):
    dP = np.zeros((ng, K))
    slope = np.zeros((ng, K))
    for g in range(ng):
        pts = np.linspace(Pmin[g], Pmax[g], K+1)
        dP[g,:] = np.diff(pts)
        C = a[g] + b[g]*pts + c[g]*(pts**2)
        slope[g,:] = np.diff(C) / dP[g,:]
    return dP, slope

dP_seg, slope_seg = build_pwl(Pmin, Pmax, a, b, c, K_SEG)

# Precompute gens-at-bus
gens_at_bus = {i: [] for i in range(nb)}
for g in range(ng):
    gens_at_bus[int(gen_bus[g])].append(g)

# Redispatch bounds (MW)
R = RAMP_FRAC * (Pmax - Pmin)

# =====================================================
# MODEL
# =====================================================
m = pyo.ConcreteModel()
m.S = pyo.Set(initialize=S_list, ordered=True)
m.B = pyo.RangeSet(0, nb-1)
m.G = pyo.RangeSet(0, ng-1)
m.K = pyo.RangeSet(0, K_SEG-1)
m.L = pyo.RangeSet(0, nl-1)

m.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)

# Base-case PWL segment vars -> Pg0
m.xseg0 = pyo.Var(m.G, m.K, domain=pyo.NonNegativeReals)

def seg_ub0(mdl, g, k):
    return mdl.xseg0[g, k] <= float(dP_seg[g, k])
m.seg_ub0 = pyo.Constraint(m.G, m.K, rule=seg_ub0)

def Pg0_rule(mdl, g):
    return float(Pmin[g]) + sum(mdl.xseg0[g, k] for k in mdl.K)
m.Pg0 = pyo.Expression(m.G, rule=Pg0_rule)


# Scenario generator outputs Pg[g,s]  <-- MUST exist before Pg_link constraint
m.Pg = pyo.Var(m.G, m.S)

# Scenario PWL segments for Pg[g,s] (all scenarios, including base)
m.xseg = pyo.Var(m.G, m.K, m.S, domain=pyo.NonNegativeReals)

def seg_ub_s(mdl, g, k, s):
    return mdl.xseg[g, k, s] <= float(dP_seg[g, k])
m.seg_ub_s = pyo.Constraint(m.G, m.K, m.S, rule=seg_ub_s)

# Link Pg[g,s] to PWL segments
def Pg_link(mdl, g, s):
    return mdl.Pg[g, s] == float(Pmin[g]) + sum(mdl.xseg[g, k, s] for k in mdl.K)
m.Pg_link = pyo.Constraint(m.G, m.S, rule=Pg_link)

# Force base-case segments to match base-case xseg0 (consistency)
def base_seg_link(mdl, g, k):
    return mdl.xseg[g, k, 0] == mdl.xseg0[g, k]
m.base_seg_link = pyo.Constraint(m.G, m.K, rule=base_seg_link)


# Angles by scenario
m.theta = pyo.Var(m.B, m.S)

# Slack angle in every scenario
for s in S_list:
    m.theta[slack_i, s].fix(0.0)

# Link base scenario Pg to Pg0
#def base_link(mdl, g):
#    return mdl.Pg[g, 0] == mdl.Pg0[g]
#m.base_link = pyo.Constraint(m.G, rule=base_link)

# Generator limits for all scenarios
def gen_bounds(mdl, g, s):
    return (float(Pmin[g]), mdl.Pg[g, s], float(Pmax[g]))
m.gen_bounds = pyo.Constraint(m.G, m.S, rule=gen_bounds)


cont_scen = [s for s in S_list if s != 0]
if len(cont_scen) == 0:
    raise RuntimeError("No contingency scenarios available (only base).")

if PROB is None:
    p = 1.0 / len(cont_scen)
    prob = {s: p for s in cont_scen}
else:
    # user-provided: must define all s>0 and sum to 1
    prob = dict(PROB)
    missing = [s for s in cont_scen if s not in prob]
    if missing:
        raise ValueError(f"Missing probabilities for scenarios: {missing}")
    psum = sum(prob[s] for s in cont_scen)
    if abs(psum - 1.0) > 1e-6:
        raise ValueError(f"Probabilities must sum to 1. Got {psum}.")


# Corrective redispatch bounds for contingencies
# |Pg[g,s] - Pg0[g]| <= R[g]
def ramp_up(mdl, g, s):
    if s == 0:
        return pyo.Constraint.Skip
    return mdl.Pg[g, s] - mdl.Pg0[g] <= float(R[g])

def ramp_dn(mdl, g, s):
    if s == 0:
        return pyo.Constraint.Skip
    return mdl.Pg0[g] - mdl.Pg[g, s] <= float(R[g])

m.ramp_up = pyo.Constraint(m.G, m.S, rule=ramp_up)
m.ramp_dn = pyo.Constraint(m.G, m.S, rule=ramp_dn)

# Power balance per bus per scenario (ENS=0 here)
def balance(mdl, i, s):
    Pg_i = sum(mdl.Pg[g, s] for g in gens_at_bus[i]) if gens_at_bus[i] else 0.0
    rhs = BASE_MVA * sum(float(Bbus_s[s][i, j]) * mdl.theta[j, s] for j in range(nb))
    return Pg_i - float(Pd[i]) == rhs

m.balance = pyo.Constraint(m.B, m.S, rule=balance)

# Line flows + limits (skip outaged line in scenario)
def flow_expr(mdl, l, s):
    return BASE_MVA * (mdl.theta[int(f_bus[l]), s] - mdl.theta[int(t_bus[l]), s]) / float(x[l])
m.F = pyo.Expression(m.L, m.S, rule=flow_expr)

m.flow_ub = pyo.ConstraintList()
m.flow_lb = pyo.ConstraintList()
m.out0 = pyo.ConstraintList()

for s in S_list:
    out_l = scenario_outage[s]
    for l in range(nl):
        if out_l is not None and l == out_l:
            m.out0.add(m.F[l, s] == 0.0)
        else:
            m.flow_ub.add(m.F[l, s] <= float(fmax[l]))
            m.flow_lb.add(-m.F[l, s] <= float(fmax[l]))

# Objective: PWL cost in base scenario (omit constant terms)
C0 = sum(float(slope_seg[g, k]) * m.xseg0[g, k] for g in range(ng) for k in range(K_SEG))

# PWL cost in each contingency scenario
Cs = {s: sum(float(slope_seg[g, k]) * m.xseg[g, k, s] for g in range(ng) for k in range(K_SEG))
      for s in cont_scen}

# Expected redispatch increment
E_dC = sum(float(prob[s]) * (Cs[s] - C0) for s in cont_scen)

m.obj = pyo.Objective(expr=C0 + float(ALPHA_RED)*E_dC, sense=pyo.minimize)

# =====================================================
# SOLVE
# =====================================================
solver = pyo.SolverFactory("highs")
res = solver.solve(m, tee=False, load_solutions=False)
term = str(res.solver.termination_condition).lower()
print("Solver:", res.solver.status, res.solver.termination_condition)
if "optimal" not in term and "feasible" not in term:
    raise RuntimeError("Corrective SCOPF not solved.")
m.solutions.load_from(res)

# =====================================================
# EXTRACT RESULTS
# =====================================================
Pg0 = np.array([pyo.value(m.Pg0[g]) for g in range(ng)], dtype=float)
Pg_s = {s: np.array([pyo.value(m.Pg[g, s]) for g in range(ng)], dtype=float) for s in S_list}

theta0 = np.array([pyo.value(m.theta[i, 0]) for i in range(nb)], dtype=float)
LMP_base = np.array([-m.dual[m.balance[i, 0]] for i in range(nb)], dtype=float)

# Generator cost reporting (quadratic) for base dispatch
total_cost = float(np.sum(a + b*Pg0 + c*(Pg0**2)))
gen_rows = []
for g in range(ng):
    gen_rows.append([g, int2ext[int(gen_bus[g])], Pg0[g], float(a[g] + b[g]*Pg0[g] + c[g]*(Pg0[g]**2))])
gen_table = pd.DataFrame(gen_rows, columns=["Gen","Bus","Pg0 (MW)","Quadratic cost ($/h)"])

print("\n===== BASE DISPATCH (Corrective SCOPF) =====")
display(gen_table)
print(f"TOTAL SYSTEM COST (reported quadratic) = {total_cost:.2f} $/h")

print("\n===== BASE-CASE LMP TABLE =====")
lmp_table = pd.DataFrame({
    "Bus": [int2ext[i] for i in range(nb)],
    "Theta_base (rad)": theta0,
    "LMP_base ($/MW)": LMP_base
})
display(lmp_table)

# Redispatch table across scenarios
rows = []
for s in S_list:
    if s == 0:
        label = "BASE"
    else:
        out_l = scenario_outage[s]
        label = f"OUT_{out_l} ({int(line_df.iloc[out_l]['from'])}-{int(line_df.iloc[out_l]['to'])})"
    row = {"Scenario": s, "Label": label}
    for g in range(ng):
        row[f"Pg_gen{g}"] = Pg_s[s][g]
        row[f"dPg_gen{g}"] = Pg_s[s][g] - Pg0[g]
    rows.append(row)

dispatch_scen_table = pd.DataFrame(rows)
print("\n===== SCENARIO DISPATCH + REDISPATCH (Corrective) =====")
display(dispatch_scen_table)

# Scenario-wise line limit flags
tol = 1e-6
flag_rows = []
for s in S_list:
    out_l = scenario_outage[s]
    scen_label = "BASE" if s == 0 else f"OUTAGE line {out_l}"
    for l in range(nl):
        flow = float(pyo.value(m.F[l, s]))
        lim = float(fmax[l])
        if out_l is not None and l == out_l:
            within = "OUTAGED"
        else:
            within = "YES" if abs(flow) <= lim + tol else "NO"
        flag_rows.append([s, scen_label, l, int2ext[int(f_bus[l])], int2ext[int(t_bus[l])], flow, lim, within])

line_flags = pd.DataFrame(flag_rows, columns=["Scenario","ScenarioLabel","Line","From","To","Flow (MW)","Limit (MW)","Within?"])
print("\n===== SCENARIO-WISE LINE LIMIT FLAGS (Corrective) =====")
display(line_flags)

Corrective SCOPF scenarios (including base): 5
Outages kept: [0, 3, 9, 10]
Solver: ok optimal

===== BASE DISPATCH (Corrective SCOPF) =====


,Gen,Bus,Pg0 (MW),Quadratic cost ($/h)
0,0,1,50.0,809.87500
1,1,2,205.0,2691.86725
2,2,3,45.0,742.49025


TOTAL SYSTEM COST (reported quadratic) = 4244.23 $/h

===== BASE-CASE LMP TABLE =====


,Bus,Theta_base (rad),LMP_base ($/MW)
0,1,0.000000,0.0
1,2,0.020577,0.0
2,3,-0.038991,0.0
3,4,-0.058654,0.0
4,5,-0.092885,0.0
5,6,-0.087089,0.0



===== SCENARIO DISPATCH + REDISPATCH (Corrective) =====


,Scenario,Label,Pg_gen0,dPg_gen0,Pg_gen1,dPg_gen1,Pg_gen2,dPg_gen2
0,0,BASE,50.000000,0.000000,205.000000,0.000000,45.000000,0.000000
1,1,OUT_0 (1-2),62.750378,12.750378,129.249622,-75.750378,108.000000,63.000000
2,2,OUT_3 (2-3),52.702597,2.702597,142.500000,-62.500000,104.797403,59.797403
3,3,OUT_9 (4-5),50.000000,0.000000,92.499459,-112.500541,157.500541,112.500541
4,4,OUT_10 (5-6),120.000000,70.000000,106.462541,-98.537459,73.537459,28.537459



===== SCENARIO-WISE LINE LIMIT FLAGS (Corrective) =====


,Scenario,ScenarioLabel,Line,From,To,Flow (MW),Limit (MW),Within?
0,0,BASE,0,1,2,-10.288472,200.0,YES
1,0,BASE,1,1,4,29.326920,200.0,YES
2,0,BASE,2,1,5,30.961552,200.0,YES
3,0,BASE,3,2,3,23.827065,120.0,YES
4,0,BASE,4,2,4,79.230784,120.0,YES
5,0,BASE,5,2,5,37.820534,120.0,YES
6,0,BASE,6,2,6,53.833144,120.0,YES
7,0,BASE,7,3,5,20.728438,120.0,YES
8,0,BASE,8,3,6,48.098627,120.0,YES
9,0,BASE,9,4,5,8.557704,120.0,YES
